# Explore Polymarket history
Install `requirements-sandbox.txt` from the repository root, then select the repository's `.venv` kernel. This notebook loads saved data and runs offline. The final cell shows optional fresh-data calls.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from polytrader.data import load_history, fetch_price_history

# Works when the kernel starts in the repo root or sandbox/.
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / 'src' / 'polytrader').is_dir()), None)
if root is None:
    raise RuntimeError('Start this notebook from the polytrader repository.')
root

In [ ]:
history_path = root / 'data' / 'lepen-history-7d.json'
if not history_path.exists():
    available = sorted((root / 'data').glob('*.json'))
    if not available:
        raise FileNotFoundError('Save a history JSON under data/ first; see the README.')
    history_path = available[0]

print(f'Loading: {history_path.name}')
history = load_history(history_path)
print(f'{len(history.data)} observations')
history.metadata

In [ ]:
df = history.to_frame()
df.head()

The index uses UTC; the original Unix-second `timestamp` remains a column. Check actual coverage and resolutions before comparing observations. The conversion does not fill gaps or resample. A resolution of zero can represent a settlement point.

In [ ]:
print('First observation:', df.index.min())
print('Last observation: ', df.index.max())
df['resolution_seconds'].value_counts().sort_index()

In [ ]:
if df.empty:
    print('No observations to plot.')
else:
    title = history.metadata.get('market', {}).get('question', history_path.stem)
    ax = df['price'].plot(style='.', figsize=(10, 4), title=title)
    ax.set(xlabel='Time (UTC)', ylabel='Outcome price', ylim=(0, 1))
    plt.tight_layout()
    plt.show()

## Optional: fetch fresh data in Python
Uncomment one call below when you want to contact Polymarket. Use `days` OR `start`/`end`; dates use UTC and the end is exclusive. Saving is optional and existing files are protected from overwrite. These functions also work in regular Python scripts and future `polytrader.bot` modules.

In [ ]:
# recent = fetch_price_history(
#     token_id=history.metadata['token_id'], days=7, bucket_seconds=300,
# )
# recent.to_frame().head()

# dated = fetch_price_history(
#     token_id=history.metadata['token_id'],
#     start='2026-09-01', end='2026-09-08',
# )
# dated.save(root / 'data' / 'new-history.json')